In [93]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization,
    DepthwiseConv2D, GlobalAveragePooling2D, Input
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from imblearn.over_sampling import SMOTE

In [94]:
dataset_path = "/kaggle/input/datasets/sabahesaraki/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT"

class_counts = {}

for category in os.listdir(dataset_path):
    if category.startswith('.'):
        continue  
    category_path = os.path.join(dataset_path, category)
    if os.path.isdir(category_path):

        images = [
            img for img in os.listdir(category_path)
            if "mask" not in img.lower()
        ]
        class_counts[category] = len(images)

print("Dataset Distribution")
for k, v in class_counts.items():
    print(f"{k} : {v}")

Dataset Distribution
benign : 437
normal : 133
malignant : 210


In [95]:
IMG_SIZE = 128

images = []
labels = []

classes = ["benign", "malignant", "normal"]

for label, category in enumerate(classes):
    folder_path = os.path.join(dataset_path, category)
    for img_name in os.listdir(folder_path):
        if "mask" in img_name.lower():
            continue
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0

        images.append(img)
        labels.append(label)

images = np.array(images)
labels = np.array(labels)

print("Dataset shape:", images.shape)
print("Labels shape:", labels.shape)

Dataset shape: (780, 128, 128, 3)
Labels shape: (780,)


In [96]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, stratify=labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [97]:
def depthwise_separable_block(x, filters, strides=(1,1)):
    x = DepthwiseConv2D(kernel_size=(3,3), strides=strides, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = Conv2D(filters, (1,1), padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    return x

In [98]:
def create_full_model():
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = depthwise_separable_block(inputs, 32)
    x = MaxPooling2D()(x)

    x = depthwise_separable_block(x, 64)
    x = MaxPooling2D()(x)

    x = depthwise_separable_block(x, 128)
    x = MaxPooling2D()(x)

    x = depthwise_separable_block(x, 256)

    x = GlobalAveragePooling2D()(x)

    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)

    outputs = Dense(3, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model

In [99]:
full_model = create_full_model()

full_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

full_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weights
)

Epoch 1/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 21s 554ms/step - accuracy: 0.3044 - loss: 1.1523 - val_accuracy: 0.1709 - val_loss: 1.1000
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3943 - loss: 1.1243 - val_accuracy: 0.2735 - val_loss: 1.1091
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3933 - loss: 1.0742 - val_accuracy: 0.2735 - val_loss: 1.1156
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4292 - loss: 1.1142 - val_accuracy: 0.2735 - val_loss: 1.1293
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4993 - loss: 1.0455 - val_accuracy: 0.2735 - val_loss: 1.1391
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5534 - loss: 1.0172 - val_accuracy: 0.2735 - val_loss: 1.1563
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5612 - loss: 1.0325 - val_accuracy: 0.2735 - val_loss: 1.1828
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4950 - loss: 1.0636 - val_accuracy: 0.2735 -

In [100]:
feature_extractor = Model(
    inputs=full_model.input,
    outputs=full_model.layers[-3].output
)

train_features = feature_extractor.predict(X_train, batch_size=32, verbose=1)
val_features = feature_extractor.predict(X_val, batch_size=32, verbose=1)
test_features = feature_extractor.predict(X_test, batch_size=32, verbose=1)

18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 442ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 


In [101]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

In [102]:
baseline_model = Sequential([
    Dense(128, activation='relu', input_shape=(train_features.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

baseline_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

baseline_model.fit(
    train_features, y_train,
    validation_data=(val_features, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weights
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - accuracy: 0.3270 - loss: 1.1176 - val_accuracy: 0.1966 - val_loss: 1.0921
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3590 - loss: 1.1299 - val_accuracy: 0.2735 - val_loss: 1.1070
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2785 - loss: 1.1734 - val_accuracy: 0.2735 - val_loss: 1.0716
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4718 - loss: 1.1293 - val_accuracy: 0.5556 - val_loss: 1.0443
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4605 - loss: 1.1035 - val_accuracy: 0.4872 - val_loss: 1.0947
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2496 - loss: 1.1087 - val_accuracy: 0.2821 - val_loss: 1.1025
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2931 - loss: 1.0901 - val_accuracy: 0.6154 - val_loss: 1.0748
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3377 - loss: 1.1139 - val_accuracy: 0.2137 - val_loss: 1.0947
Ep

In [103]:
y_pred_baseline = baseline_model.predict(test_features, verbose=0)
y_pred_baseline = np.argmax(y_pred_baseline, axis=1)

baseline_acc = accuracy_score(y_test, y_pred_baseline)
baseline_pre = precision_score(y_test, y_pred_baseline, average="weighted")
baseline_rec = recall_score(y_test, y_pred_baseline, average="weighted")
baseline_f1 = f1_score(y_test, y_pred_baseline, average="weighted")

print("Baseline Accuracy :", baseline_acc)
print("Baseline Precision:", baseline_pre)
print("Baseline Recall   :", baseline_rec)
print("Baseline F1 Score :", baseline_f1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_baseline))

Baseline Accuracy : 0.17094017094017094
Baseline Precision: 0.029220542041054863
Baseline Recall   : 0.17094017094017094
Baseline F1 Score : 0.049909538960633854

Classification Report:

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        66
           1       0.00      0.00      0.00        31
           2       0.17      1.00      0.29        20

    accuracy                           0.17       117
   macro avg       0.06      0.33      0.10       117
weighted avg       0.03      0.17      0.05       117



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [104]:
smote = SMOTE(random_state=42, k_neighbors=1)

train_features_smote, y_train_smote = smote.fit_resample(
    train_features, y_train
)

In [105]:
smote_model = Sequential([
    Dense(128, activation='relu', input_shape=(train_features.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

smote_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

smote_model.fit(
    train_features_smote, y_train_smote,
    validation_data=(val_features, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


29/29 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.3098 - loss: 1.1398 - val_accuracy: 0.5556 - val_loss: 1.0711
Epoch 2/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3344 - loss: 1.1064 - val_accuracy: 0.3162 - val_loss: 1.0904
Epoch 3/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3395 - loss: 1.1016 - val_accuracy: 0.1709 - val_loss: 1.0977
Epoch 4/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3640 - loss: 1.1029 - val_accuracy: 0.1709 - val_loss: 1.0998
Epoch 5/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3634 - loss: 1.0943 - val_accuracy: 0.5726 - val_loss: 1.0822
Epoch 6/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3057 - loss: 1.1041 - val_accuracy: 0.2393 - val_loss: 1.0975
Epoch 7/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3384 - loss: 1.1003 - val_accuracy: 0.5726 - val_loss: 1.0671
Epoch 8/20
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3599 - loss: 1.0943 - val_accuracy: 0.5983 - val_loss: 1.0769
Ep

In [106]:
y_pred_smote = smote_model.predict(test_features, verbose=0)
y_pred_smote = np.argmax(y_pred_smote, axis=1)

smote_acc = accuracy_score(y_test, y_pred_smote)
smote_pre = precision_score(y_test, y_pred_smote, average="weighted")
smote_rec = recall_score(y_test, y_pred_smote, average="weighted")
smote_f1 = f1_score(y_test, y_pred_smote, average="weighted")

print("SMOTE Accuracy :", smote_acc)
print("SMOTE Precision:", smote_pre)
print("SMOTE Recall   :", smote_rec)
print("SMOTE F1 Score :", smote_f1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_smote))

SMOTE Accuracy : 0.49572649572649574
SMOTE Precision: 0.48381952186300015
SMOTE Recall   : 0.49572649572649574
SMOTE F1 Score : 0.4590643274853801

Classification Report:

              precision    recall  f1-score   support

           0       0.69      0.50      0.58        66
           1       0.36      0.81      0.50        31
           2       0.00      0.00      0.00        20

    accuracy                           0.50       117
   macro avg       0.35      0.44      0.36       117
weighted avg       0.48      0.50      0.46       117



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [107]:
aug_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

augmented_images = []
augmented_labels = []

for img, label in zip(X_train, y_train):
    img = img.reshape((1, IMG_SIZE, IMG_SIZE, 3))
    aug_iter = aug_datagen.flow(img, batch_size=1)

    aug_img = next(aug_iter)[0]
    augmented_images.append(aug_img)
    augmented_labels.append(label)

augmented_images = np.array(augmented_images)
augmented_labels = np.array(augmented_labels)

In [108]:
X_train_aug = np.concatenate((X_train, augmented_images), axis=0)
y_train_aug = np.concatenate((y_train, augmented_labels), axis=0)

In [109]:
X_train_aug = np.concatenate((X_train, augmented_images), axis=0)
y_train_aug = np.concatenate((y_train, augmented_labels), axis=0)

print("Combined training data:", X_train_aug.shape)
print("Combined labels:", y_train_aug.shape)

train_features_aug = feature_extractor.predict(X_train_aug, batch_size=32, verbose=1)

Combined training data: (1092, 128, 128, 3)
Combined labels: (1092,)
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step


In [110]:
class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_aug),
    y=y_train_aug
)

class_weights = {
    cls: weight for cls, weight in zip(np.unique(y_train_aug), class_weights_values)
}
print("Class weights:", class_weights)

Class weights: {np.int64(0): np.float64(0.5947712418300654), np.int64(1): np.float64(1.2380952380952381), np.int64(2): np.float64(1.956989247311828)}


In [111]:
aug_model = Sequential([
    Dense(128, activation='relu', input_shape=(train_features.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

aug_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_model.fit(
    train_features_aug, y_train_aug,
    validation_data=(val_features, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.4836 - loss: 1.0529 - val_accuracy: 0.5556 - val_loss: 0.9880
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5668 - loss: 0.9749 - val_accuracy: 0.5556 - val_loss: 0.9876
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5594 - loss: 1.0000 - val_accuracy: 0.5556 - val_loss: 0.9840
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5363 - loss: 0.9921 - val_accuracy: 0.5556 - val_loss: 0.9840
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5574 - loss: 0.9851 - val_accuracy: 0.5556 - val_loss: 0.9798
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5567 - loss: 0.9940 - val_accuracy: 0.5556 - val_loss: 0.9797
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5739 - loss: 0.9754 - val_accuracy: 0.5556 - val_loss: 0.9843
Epoch 8/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5505 - loss: 0.9983 - val_accuracy: 0.5556 - val_loss: 0.9804
Ep

In [112]:
y_pred_aug = aug_model.predict(test_features, verbose=0)
y_pred_aug = np.argmax(y_pred_aug, axis=1)

aug_acc = accuracy_score(y_test, y_pred_aug)
aug_pre = precision_score(y_test, y_pred_aug, average="weighted")
aug_rec = recall_score(y_test, y_pred_aug, average="weighted")
aug_f1 = f1_score(y_test, y_pred_aug, average="weighted")

print("Augmentation + Class Weight Accuracy :", aug_acc)
print("Precision:", aug_pre)
print("Recall   :", aug_rec)
print("F1 Score :", aug_f1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_aug))

Augmentation + Class Weight Accuracy : 0.5641025641025641
Precision: 0.31821170282708744
Recall   : 0.5641025641025641
F1 Score : 0.40689365279529216

Classification Report:

              precision    recall  f1-score   support

           0       0.56      1.00      0.72        66
           1       0.00      0.00      0.00        31
           2       0.00      0.00      0.00        20

    accuracy                           0.56       117
   macro avg       0.19      0.33      0.24       117
weighted avg       0.32      0.56      0.41       117



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [113]:
print("Baseline Accuracy:", baseline_acc)
print("SMOTE Accuracy:", smote_acc)
print("Augmentation + Class Weight Accuracy:", aug_acc)

Baseline Accuracy: 0.17094017094017094
SMOTE Accuracy: 0.49572649572649574
Augmentation + Class Weight Accuracy: 0.5641025641025641
